Rasterio - Reads and writes geospatial raster files (satellite images, DEMs, GeoTIFFs)

GeoPandas - Pandas extension for working with geospatial vector data (shapefiles, polygons, points)

Shapely - Creates and manipulates geometric shapes (points, lines, polygons)

Pyproj - Converts between different coordinate systems and map projections

Matplotlib - Creates plots, charts, and visualizations

NumPy - Fast array operations and mathematical computations

SciPy - Advanced scientific computing (statistics, signal processing, filters)

In [ ]:
# Required libraries
!pip install rasterio geopandas shapely pyproj matplotlib numpy scipy

In [ ]:
import os
import urllib.request

os.makedirs("data", exist_ok=True)
# URL of data from nasa's site
dem_url = (
    "https://planetarymaps.usgs.gov/mosaic/Mars/HRSC_MOLA_Blend/Mars_HRSC_MOLA_BlendDEM_Global_200mp_v2.tif"
)

# Path
dem_path = "data/mars_global_dem_200m.tif"

# Download if not present in directory
if not os.path.exists(dem_path):
    print("Downloading Mars global DEM (this may take a few minutes)...")
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(dem_url, dem_path)
    print("Download complete.")
else:
    print("DEM already exists.")

In [2]:
import rasterio
from rasterio.windows import from_bounds

# Elysium Planitia bounding box (degrees)
lat_min, lat_max = -5.12044, -4.45392
lon_min, lon_max = 136.46052 ,137.8255

output_path = "data/elysium_planitia_dem.tif"

with rasterio.open(dem_path) as src:
    print("CRS:", src.crs)

    # Create window from geographic bounds
    window = from_bounds(
        lon_min, lat_min,
        lon_max, lat_max,
        transform=src.transform
    )

    elysium_data = src.read(1, window=window)
    elysium_transform = src.window_transform(window)

    meta = src.meta.copy()
    meta.update({
        "height": elysium_data.shape[0],
        "width": elysium_data.shape[1],
        "transform": elysium_transform
    })

    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(elysium_data, 1)

print("✅ Elysium Planitia DEM saved at:", output_path)

In [3]:
import matplotlib.pyplot as plt

with rasterio.open("data/elysium_planitia_dem.tif") as dem:
    elevation = dem.read(1)

plt.figure(figsize=(8,6))
plt.imshow(elevation, cmap="terrain")
plt.colorbar(label="Elevation (meters)")
plt.title("Elysium Planitia – Mars Elevation")
plt.axis("off")
plt.show()


In [4]:
import numpy as np
import rasterio
from scipy.ndimage import sobel

with rasterio.open("data/elysium_planitia_dem.tif") as src:
    dem = src.read(1).astype(float)
    transform = src.transform

    # Original pixel size in degrees (from the coordinate system)
    pixel_size_degrees = transform.a

    # Mars equatorial radius in meters (approximate value)
    mars_equatorial_radius_meters = 3396200.0 # 3396.2 km

    # Convert pixel size from degrees to meters
    # The length of one degree of longitude/latitude in meters at the equator
    meters_per_degree = (2 * np.pi * mars_equatorial_radius_meters) / 360.0

    # Calculate the pixel size in meters
    pixel_size = pixel_size_degrees * meters_per_degree

# Compute gradients
# The Sobel filter output from scipy.ndimage.sobel is scaled by 1/8 internally
# for integer inputs to preserve magnitude. When using float inputs,
# it is common practice to still divide by 8 * pixel_size to normalize the gradient
# into change in elevation per meter.
dzdx = sobel(dem, axis=1) / (8 * pixel_size)
dzdy = sobel(dem, axis=0) / (8 * pixel_size)

# Slope in degrees
slope = np.degrees(np.arctan(np.sqrt(dzdx**2 + dzdy**2)))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.imshow(slope, cmap="inferno")
plt.colorbar(label="Slope (degrees)")
plt.title("Mars Surface Slope – Elysium Planitia")
plt.axis("off")
plt.show()


In [ ]:
from scipy.ndimage import generic_filter

def roughness_func(window):
    return np.std(window)

roughness = generic_filter(dem, roughness_func, size=5)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(roughness, cmap="magma")
plt.colorbar(label="Elevation Std Dev (m)")
plt.title("Surface Roughness – Elysium Planitia")
plt.axis("off")
plt.show()

In [ ]:
SLOPE_THRESHOLD = 10      # degrees
ROUGHNESS_THRESHOLD = 10  # meters (tunable)

safe_mask = (slope <= SLOPE_THRESHOLD) & (roughness <= ROUGHNESS_THRESHOLD)

labels = np.zeros_like(dem, dtype=np.uint8)
labels[safe_mask] = 1

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(labels, cmap="Greens")
plt.title("Safe Landing Zones (Green)")
plt.axis("off")
plt.show()

In [ ]:
labels_multi = np.zeros_like(dem, dtype=np.uint8)

# SAFE
labels_multi[(slope <= 10) & (roughness <= 2)] = 2

# MODERATE
labels_multi[
    ((slope > 5) & (slope <= 10)) |
    ((roughness > 1.5) & (roughness <= 3))
] = 1

# HAZARD remains 0

In [ ]:
import matplotlib.colors as colors

cmap = colors.ListedColormap(["green", "orange", "red"])
bounds = [0, 1, 2, 3]
norm = colors.BoundaryNorm(bounds, cmap.N)

plt.figure(figsize=(9,5))
plt.imshow(labels_multi, cmap=cmap, norm=norm)
plt.title("Mars Landing Safety Map (Elysium Planitia)")
plt.axis("off")
plt.show()

#### The Deeplearning phase

In [ ]:
import numpy as np

input_stack = np.stack([dem, slope, roughness], axis=0)

print("Input shape:", input_stack.shape)
print("Label shape:", labels_multi.shape)

In [ ]:
# patchs for training
PATCH_SIZE = 256
stride = 256

patches = []
masks = []

C, H, W = input_stack_norm.shape

for i in range(0, H - PATCH_SIZE, stride):
    for j in range(0, W - PATCH_SIZE, stride):
        patch = input_stack_norm[:, i:i+PATCH_SIZE, j:j+PATCH_SIZE]
        mask = labels_multi[i:i+PATCH_SIZE, j:j+PATCH_SIZE]

        patches.append(patch)
        masks.append(mask)

patches = np.array(patches)
masks = np.array(masks)

print("Total patches:", len(patches))
print("Patch shape:", patches.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    patches, masks, test_size=0.2, random_state=42
)

print("Train samples:", len(X_train))
print("Validation samples:", len(X_val))

In [ ]:
# Tourch Installation
!pip install torch torchvision

In [ ]:
# Dataset Class

import torch
from torch.utils.data import Dataset

class MarsDataset(Dataset):
    def __init__(self, images, masks):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.masks = torch.tensor(masks, dtype=torch.long)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]

train_dataset = MarsDataset(X_train, y_train)
val_dataset = MarsDataset(X_val, y_val)

In [ ]:
# loading
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(UNet, self).__init__()

        self.enc1 = nn.Conv2d(in_channels, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.dec1 = nn.Conv2d(64, 32, 3, padding=1)
        self.final = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        x1 = F.relu(self.enc1(x))
        x2 = self.pool(x1)
        x3 = F.relu(self.enc2(x2))

        x4 = F.interpolate(x3, scale_factor=2, mode='bilinear', align_corners=False)
        x5 = F.relu(self.dec1(x4))

        return self.final(x5)

model = UNet()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
# 5 epochs
for epoch in range(5):
    model.train()
    total_loss = 0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")